In [1]:
import numpy as np
import pandas as pd

## Phase 2 : Data Cleaning and Portfolio Construction ( Buy and Hold )

In [2]:
# Step 0: Loading Phase 1 outputs
prices = pd.read_csv(r'C:\Users\sentr\Downloads\Internships\QFI\CSV Files\Raw_Portfolio_Price_stratified.csv', index_col = 0, parse_dates = True)
benchmark_df = pd.read_csv(r'C:\Users\sentr\Downloads\Internships\QFI\CSV Files\Benchmark_prices_1.csv', index_col = 0, parse_dates = True)
# Single column: Plain Series
benchmark_prices = benchmark_df.iloc[:,0]
print(f'Loaded {prices.shape[1]} tickers and {len(prices)} raw rows')

Loaded 50 tickers and 1507 raw rows


In [3]:
# Step 1: Establishing a common usable window 
first_valid_dates = prices.apply(lambda col: col.first_valid_index())
common_start = first_valid_dates.max()
print(f'Latest first valid date among all tickers {common_start.date()}')

laggards = first_valid_dates[first_valid_dates == common_start]
print('Tickers that set this cutoff')
print(laggards)

trimmed_prices = prices.loc[common_start:]
trimmed_benchmark = benchmark_prices.loc[common_start:]


Latest first valid date among all tickers 2020-07-24
Tickers that set this cutoff
AFL     2020-07-24
AON     2020-07-24
APA     2020-07-24
APD     2020-07-24
BA      2020-07-24
BKNG    2020-07-24
BKR     2020-07-24
CI      2020-07-24
CINF    2020-07-24
CMI     2020-07-24
COF     2020-07-24
D       2020-07-24
DVA     2020-07-24
EA      2020-07-24
ECL     2020-07-24
ED      2020-07-24
EQR     2020-07-24
ESS     2020-07-24
ETN     2020-07-24
GM      2020-07-24
GOOGL   2020-07-24
HST     2020-07-24
INTU    2020-07-24
KLAC    2020-07-24
KMB     2020-07-24
KR      2020-07-24
L       2020-07-24
LLY     2020-07-24
LRCX    2020-07-24
MAS     2020-07-24
MLM     2020-07-24
MMM     2020-07-24
NDAQ    2020-07-24
NOW     2020-07-24
PM      2020-07-24
PWR     2020-07-24
RF      2020-07-24
ROP     2020-07-24
ROST    2020-07-24
RVTY    2020-07-24
SO      2020-07-24
TFC     2020-07-24
TGT     2020-07-24
TPR     2020-07-24
TXN     2020-07-24
TXT     2020-07-24
ULTA    2020-07-24
URI     2020-07-24
VRTX  

In [4]:
# Step 2 : Defensive Imputation (Fill only small gaps)
print('--- Missing Value Check and Defensive Imputation ---')
# Checking the current status of the data
initial_missing = trimmed_prices.isna().sum().sum() == 0
if initial_missing == 0:
    print('There is no missing value in the current historical window')
    print('Executing bounded forward fill ( limit = 3) regardless as a defensive measure for future live data')
else:
    print('Found {len(initial_missing)} missing data points. Executing bounded forward fill')

# Applying forward fill
ffill_limit = 3
filled_price = trimmed_prices.ffill(limit = ffill_limit)

remaining_missing = filled_price.isna().sum()
tickers_with_gaps = remaining_missing[remaining_missing > 0]

if len(tickers_with_gaps) > 0:
    print(f'{len(tickers_with_gaps)} ticker(s) still have gaps longer than '
          f'{ffill_limit} days after filling. Excluding these tickers rather than '
          f'dropping shared rows for everyone else:')
    print(tickers_with_gaps)
    clean_price = filled_price.drop(columns = tickers_with_gaps.index.tolist())
else:
    print(f'\nNo tickers had gaps longer than {ffill_limit} days. None excluded.')
    clean_price = filled_price

clean_benchmark = trimmed_benchmark.ffill(limit = ffill_limit).dropna()

print(f'Final clean universe: {clean_price.shape[1]} tickers, {len(clean_price)} trading days')

--- Missing Value Check and Defensive Imputation ---
Found {len(initial_missing)} missing data points. Executing bounded forward fill

No tickers had gaps longer than 3 days. None excluded.
Final clean universe: 50 tickers, 1507 trading days


In [5]:
# Flagging stale price and implausible moves
def max_consecutive_repeats(series):
    is_repeat = series == series.shift(1)
    run_id = (~is_repeat).cumsum()
    return is_repeat.groupby(run_id).sum().max()

Stale_Run_Threshold = 5
stale_run = clean_price.apply(max_consecutive_repeats)
flagged_stale = stale_run[stale_run >= Stale_Run_Threshold]
if len(flagged_stale) > 0:
    print(f"\nMONITOR (not excluded) - ticker(s) with {Stale_Run_Threshold}+ identical "
          f"consecutive prices (possible stale/no-trade days):")
    print(flagged_stale)

check_return = clean_prices.pct_change()
Extreme_move_threshold = 0.4 # 40% single day change. Flaging for error check. Don't autoclip
